# TKCE — Deep-dive: training dynamics (standalone)

Trains **TKCE-joint** on ONE balanced binary dataset (`eye_movements`, 50/50) with a
**much deeper** Siamese encoder + TabResNet, **600 epochs**, a **slow learning rate**,
and **no early stopping** — then plots **loss** and **AUC/accuracy per epoch** for both
the **training** and **validation** sets.

**This notebook is independent of the main suite** — you can run it in a separate
session while the main run continues.

**Setup:** `Runtime → Change runtime type → A100 (or H100 / L4) GPU`. This experiment
is pure neural-network training, so a strong GPU really does speed it up.
Expected time: ~10–40 min depending on GPU.

## 1. Config — edit only if you want different settings

In [ ]:
REPO_URL   = "https://github.com/sushanedulloo/TKCE.git"
DRIVE_BASE = "/content/drive/MyDrive/tkce_results"

# ---- deep + long + SLOW settings ----
TASK      = 361070      # eye_movements: 7,608 rows / 20 features, balanced
SEEDS     = "0,1,2"     # shuffles (new random split each)
EPOCHS    = 600
LR        = 1e-6        # slow: training was reaching AUC 1 too fast
BATCH     = 64          # TASK batch: small -> more updates + regularisation
CBATCH    = 256         # CONTRASTIVE batch: large -> keeps 510 negatives
ENC_WIDTH = 512
ENC_DEPTH = 6
EMB       = 128
N_BLOCKS  = 8
D         = 256
D_HIDDEN  = 512
HEAD      = "tabresnet"

# lambda sweep (contrastive acts as a REGULARIZER -> small lambda)
LAMBDAS   = "0,0.005,0.015,0.05,0.15,0.5"
LOSSES    = "infonce,aninfonce,clip_infonce,contrastive,triplet"

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
FIG = DRIVE_BASE + "/figures"
AN  = DRIVE_BASE + "/analysis"
LOG = DRIVE_BASE + "/deep_joint_run.log"
for d in (DRIVE_BASE, FIG, AN): os.makedirs(d, exist_ok=True)
print('Results will be saved to:', FIG, 'and', AN)

## 3. Get the code + install

In [ ]:
%cd /content
!rm -rf tkce_deep
!git clone $REPO_URL tkce_deep
%cd tkce_deep
!pip install -q -r requirements-tkce.txt
print('Ready')

## 4. GPU check

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
!nvidia-smi -L

## 5. LAMBDA SWEEP (run this first)
Sweeps the fused-loss weight for one loss. The contrastive term is a *regularizer*,
so the useful range is small — this finds where it helps most.

In [ ]:
REPO_URL   = "https://github.com/sushanedulloo/TKCE.git"
DRIVE_BASE = "/content/drive/MyDrive/tkce_results"

# ---- deep + long + SLOW settings ----
TASK      = 361070      # eye_movements: 7,608 rows / 20 features, balanced
SEEDS     = "0,1,2"     # shuffles (new random split each)
EPOCHS    = 600
LR        = 1e-6        # slow: training was reaching AUC 1 too fast
BATCH     = 64          # TASK batch: small -> more updates + regularisation
CBATCH    = 256         # CONTRASTIVE batch: large -> keeps 510 negatives
ENC_WIDTH = 512
ENC_DEPTH = 6
EMB       = 128
N_BLOCKS  = 8
D         = 256
D_HIDDEN  = 512
HEAD      = "tabresnet"

# lambda sweep (contrastive acts as a REGULARIZER -> small lambda)
LAMBDAS   = "0,0.005,0.015,0.05,0.15,0.5"
LOSSES    = "infonce,aninfonce,clip_infonce,contrastive,triplet"

## 6. LOSS COMPARISON at the best lambda
Set `BEST_LAMBDA` from the sweep above, then compare all 5 contrastive losses.

In [ ]:
REPO_URL   = "https://github.com/sushanedulloo/TKCE.git"
DRIVE_BASE = "/content/drive/MyDrive/tkce_results"

# ---- deep + long + SLOW settings ----
TASK      = 361070      # eye_movements: 7,608 rows / 20 features, balanced
SEEDS     = "0,1,2"     # shuffles (new random split each)
EPOCHS    = 600
LR        = 1e-6        # slow: training was reaching AUC 1 too fast
BATCH     = 64          # TASK batch: small -> more updates + regularisation
CBATCH    = 256         # CONTRASTIVE batch: large -> keeps 510 negatives
ENC_WIDTH = 512
ENC_DEPTH = 6
EMB       = 128
N_BLOCKS  = 8
D         = 256
D_HIDDEN  = 512
HEAD      = "tabresnet"

# lambda sweep (contrastive acts as a REGULARIZER -> small lambda)
LAMBDAS   = "0,0.005,0.015,0.05,0.15,0.5"
LOSSES    = "infonce,aninfonce,clip_infonce,contrastive,triplet"

## 7. (Optional) Let the model LEARN the balance
Uncertainty weighting instead of a hand-set lambda.

In [ ]:
REPO_URL   = "https://github.com/sushanedulloo/TKCE.git"
DRIVE_BASE = "/content/drive/MyDrive/tkce_results"

# ---- deep + long + SLOW settings ----
TASK      = 361070      # eye_movements: 7,608 rows / 20 features, balanced
SEEDS     = "0,1,2"     # shuffles (new random split each)
EPOCHS    = 600
LR        = 1e-6        # slow: training was reaching AUC 1 too fast
BATCH     = 64          # TASK batch: small -> more updates + regularisation
CBATCH    = 256         # CONTRASTIVE batch: large -> keeps 510 negatives
ENC_WIDTH = 512
ENC_DEPTH = 6
EMB       = 128
N_BLOCKS  = 8
D         = 256
D_HIDDEN  = 512
HEAD      = "tabresnet"

# lambda sweep (contrastive acts as a REGULARIZER -> small lambda)
LAMBDAS   = "0,0.005,0.015,0.05,0.15,0.5"
LOSSES    = "infonce,aninfonce,clip_infonce,contrastive,triplet"

## 5. Run the deep-dive experiment
Progress prints live. It trains the full number of epochs (no early stopping) so the
complete trend is visible.

In [ ]:
REPO_URL   = "https://github.com/sushanedulloo/TKCE.git"
DRIVE_BASE = "/content/drive/MyDrive/tkce_results"

# ---- deep + long + SLOW settings ----
TASK      = 361070      # eye_movements: 7,608 rows / 20 features, balanced
SEEDS     = "0,1,2"     # shuffles (new random split each)
EPOCHS    = 600
LR        = 1e-6        # slow: training was reaching AUC 1 too fast
BATCH     = 64          # TASK batch: small -> more updates + regularisation
CBATCH    = 256         # CONTRASTIVE batch: large -> keeps 510 negatives
ENC_WIDTH = 512
ENC_DEPTH = 6
EMB       = 128
N_BLOCKS  = 8
D         = 256
D_HIDDEN  = 512
HEAD      = "tabresnet"

# lambda sweep (contrastive acts as a REGULARIZER -> small lambda)
LAMBDAS   = "0,0.005,0.015,0.05,0.15,0.5"
LOSSES    = "infonce,aninfonce,clip_infonce,contrastive,triplet"

## 5b. Same run, but with BALANCED loss weighting
InfoNCE is ~7x larger than cross-entropy, so with the default $\lambda$ the task loss is
**subdued** (the contrastive term dominates the gradient). `--balance-losses` rescales
$\lambda$ so both terms start at equal magnitude. Run this and compare the two figures.

## 5c. Same run, but the MODEL LEARNS the balance
Uncertainty weighting (Kendall et al. 2018): instead of you choosing $\lambda$, a
trainable weight per loss is learned jointly with the network. The figure's last
panel then shows the balance the model **chose by itself** over training.

In [ ]:
REPO_URL   = "https://github.com/sushanedulloo/TKCE.git"
DRIVE_BASE = "/content/drive/MyDrive/tkce_results"

# ---- deep + long + SLOW settings ----
TASK      = 361070      # eye_movements: 7,608 rows / 20 features, balanced
SEEDS     = "0,1,2"     # shuffles (new random split each)
EPOCHS    = 600
LR        = 1e-6        # slow: training was reaching AUC 1 too fast
BATCH     = 64          # TASK batch: small -> more updates + regularisation
CBATCH    = 256         # CONTRASTIVE batch: large -> keeps 510 negatives
ENC_WIDTH = 512
ENC_DEPTH = 6
EMB       = 128
N_BLOCKS  = 8
D         = 256
D_HIDDEN  = 512
HEAD      = "tabresnet"

# lambda sweep (contrastive acts as a REGULARIZER -> small lambda)
LAMBDAS   = "0,0.005,0.015,0.05,0.15,0.5"
LOSSES    = "infonce,aninfonce,clip_infonce,contrastive,triplet"

In [ ]:
REPO_URL   = "https://github.com/sushanedulloo/TKCE.git"
DRIVE_BASE = "/content/drive/MyDrive/tkce_results"

# ---- deep + long + SLOW settings ----
TASK      = 361070      # eye_movements: 7,608 rows / 20 features, balanced
SEEDS     = "0,1,2"     # shuffles (new random split each)
EPOCHS    = 600
LR        = 1e-6        # slow: training was reaching AUC 1 too fast
BATCH     = 64          # TASK batch: small -> more updates + regularisation
CBATCH    = 256         # CONTRASTIVE batch: large -> keeps 510 negatives
ENC_WIDTH = 512
ENC_DEPTH = 6
EMB       = 128
N_BLOCKS  = 8
D         = 256
D_HIDDEN  = 512
HEAD      = "tabresnet"

# lambda sweep (contrastive acts as a REGULARIZER -> small lambda)
LAMBDAS   = "0,0.005,0.015,0.05,0.15,0.5"
LOSSES    = "infonce,aninfonce,clip_infonce,contrastive,triplet"

## 6. Show the resulting figure

In [ ]:
from IPython.display import Image, display
import glob
pngs = sorted(glob.glob(FIG + '/deep_joint_*.png'))
print(pngs)
for p in pngs: display(Image(p))

## 7. Peek at the per-epoch numbers

In [ ]:
import pandas as pd, glob
t = sorted(glob.glob(AN + '/deep_joint_*_test.csv'))[-1]
tdf = pd.read_csv(t)
print('TEST RESULT PER SHUFFLE:'); display(tdf)
print('mean test AUC = %.4f +/- %.4f' % (tdf.test_auc.mean(), tdf.test_auc.std(ddof=0)))
e = sorted(glob.glob(AN + '/deep_joint_*_epochs.csv'))[-1]
edf = pd.read_csv(e)
print('per-epoch rows:', len(edf), '| shuffles:', edf.seed.nunique())
display(edf.tail())

## Want to go deeper / longer?
Change the settings in **cell 1** and re-run cells 1 and 5, e.g.:
- Deeper: `ENC_DEPTH = 8`, `ENC_WIDTH = 768`, `N_BLOCKS = 12`
- Longer/slower: `EPOCHS = 1000`, `LR = 5e-5`
- Compare heads: `HEAD = "mlp"`

**Note:** re-running overwrites the previous figure/CSV. To keep both, change
`DRIVE_BASE` (e.g. add `/run2`) before re-running.